# Skin Disease 0.99 Sprint (Stable Edition)

Stable-focused upgrade from the 0.9704 pipeline:

1. Multi-seed and multi-architecture member ensemble
2. OOF-based member weight optimization (Dirichlet random search)
3. OOF bias + global temperature calibration
4. Rich 8-view TTA
5. Optional conservative pseudo-label adjustment (toggle)

Design goal: improve leaderboard score while minimizing collapse risk.


In [ ]:
import copy
import gc
import json
import random
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, confusion_matrix, classification_report

from torchvision import models
from torchvision.transforms import v2

print('torch', torch.__version__)


In [ ]:
# =========================
# Config / paths
# =========================
LOCAL_BASE = Path('/Users/songling/Desktop/Skin Disease Classification')
PLATFORM_BASE = Path('dataset/public')

if PLATFORM_BASE.exists():
    MODE = 'platform'
    BASE_DIR = PLATFORM_BASE
    OUTPUT_DIR = Path('working')
elif LOCAL_BASE.exists():
    MODE = 'local'
    BASE_DIR = LOCAL_BASE
    OUTPUT_DIR = BASE_DIR
else:
    raise FileNotFoundError('Dataset path not found.')

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RUNS_DIR = OUTPUT_DIR / 'runs'
RUNS_DIR.mkdir(parents=True, exist_ok=True)
run_name = datetime.now().strftime('run_99sprint_%Y%m%d_%H%M%S')
run_dir = RUNS_DIR / run_name
run_dir.mkdir(parents=True, exist_ok=False)

TRAIN_CSV = BASE_DIR / 'train.csv'
TEST_CSV = BASE_DIR / 'test.csv'
TRAIN_IMG_DIR = BASE_DIR / 'train'
TEST_IMG_DIR = BASE_DIR / 'test'

CLASS_NAMES = ['acne', 'eksim', 'herpes', 'panu', 'rosacea']
label2idx = {c: i for i, c in enumerate(CLASS_NAMES)}
idx2label = {i: c for c, i in label2idx.items()}
N_CLASSES = len(CLASS_NAMES)

SEEDS = [42, 2024]
N_SPLITS = 5
IMG_SIZE = 224
BATCH_SIZE = 16

EPOCHS = 14
HEAD_EPOCHS = 2
LR_HEAD = 1e-3
LR_FINE = 2e-4
WEIGHT_DECAY = 1e-4
LABEL_SMOOTHING = 0.05
EARLY_STOP = 4
GRAD_CLIP_NORM = 1.0

# Conservative optional pseudo stage.
ENABLE_PSEUDO = True
PSEUDO_THRESHOLD = 0.995
PSEUDO_MIN_COUNT = 12
PSEUDO_ALPHA = 0.15

MODEL_CONFIGS = [
    {'name': 'swin_v2_t', 'base_weight': 0.45},
    {'name': 'efficientnet_v2_s', 'base_weight': 0.35},
    {'name': 'convnext_small', 'base_weight': 0.20},
]


# Filter unsupported models in runtime environment
_available_model_map = {
    'swin_v2_t': hasattr(models, 'swin_v2_t'),
    'efficientnet_v2_s': hasattr(models, 'efficientnet_v2_s'),
    'convnext_small': hasattr(models, 'convnext_small'),
}
MODEL_CONFIGS = [m for m in MODEL_CONFIGS if _available_model_map.get(m['name'], False)]
if len(MODEL_CONFIGS) == 0:
    raise RuntimeError('No supported models found in this runtime.')

NUM_WORKERS = 0 if MODE == 'local' else 2
PERSISTENT_WORKERS = NUM_WORKERS > 0

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')

PIN_MEMORY = (device.type == 'cuda')
USE_AMP = (device.type == 'cuda')

print('Mode:', MODE)
print('Base dir:', BASE_DIR)
print('Output dir:', OUTPUT_DIR)
print('Run dir:', run_dir)
print('Device:', device)



In [ ]:
# =========================
# Data / transforms
# =========================
train_df = pd.read_csv(TRAIN_CSV)
test_df = pd.read_csv(TEST_CSV)

assert set(train_df['disease'].unique()) == set(CLASS_NAMES), 'Class mismatch'
assert len(test_df) == 180, f'Test rows should be 180, got {len(test_df)}'

train_df['label'] = train_df['disease'].map(label2idx)

def make_class_weights(df):
    counts = df['label'].value_counts().sort_index().values
    weights = len(df) / (N_CLASSES * counts)
    return torch.tensor(weights, dtype=torch.float32)

class AddGaussianNoise(nn.Module):
    def __init__(self, std=0.02, p=0.25):
        super().__init__()
        self.std = std
        self.p = p

    def forward(self, x):
        if torch.rand(1).item() < self.p:
            x = torch.clamp(x + torch.randn_like(x) * self.std, 0.0, 1.0)
        return x

MEAN = [0.485, 0.456, 0.406]
STD = [0.229, 0.224, 0.225]

train_tfms = v2.Compose([
    v2.Resize((IMG_SIZE, IMG_SIZE)),
    v2.RandomHorizontalFlip(0.5),
    v2.RandomVerticalFlip(0.15),
    v2.RandomRotation(20),
    v2.RandomAffine(degrees=0, translate=(0.08, 0.08), scale=(0.9, 1.1)),
    v2.ColorJitter(brightness=0.18, contrast=0.18, saturation=0.12, hue=0.03),
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    AddGaussianNoise(std=0.02, p=0.25),
    v2.Normalize(MEAN, STD),
])

valid_tfms = v2.Compose([
    v2.Resize((IMG_SIZE, IMG_SIZE)),
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(MEAN, STD),
])

class SkinDataset(Dataset):
    def __init__(self, df, image_dir, transform=None, is_test=False):
        self.df = df.reset_index(drop=True)
        self.image_dir = Path(image_dir)
        self.transform = transform
        self.is_test = is_test

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(self.image_dir / row['filename']).convert('RGB')
        if self.transform is not None:
            img = self.transform(img)
        if self.is_test:
            return img, int(row['id'])
        return img, int(row['label'])


def build_loader(ds, shuffle):
    kwargs = dict(batch_size=BATCH_SIZE, shuffle=shuffle, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
    if PERSISTENT_WORKERS:
        kwargs['persistent_workers'] = True
    return DataLoader(ds, **kwargs)


In [ ]:
# =========================
# Model / TTA / calibration utilities
# =========================
def build_model(model_name):
    # Robust to environments where pretrained weights cannot be downloaded.
    if model_name == 'swin_v2_t':
        try:
            m = models.swin_v2_t(weights=models.Swin_V2_T_Weights.IMAGENET1K_V1)
        except Exception as e:
            print('Warning: swin_v2_t pretrained unavailable, fallback to random init:', e)
            m = models.swin_v2_t(weights=None)
        in_features = m.head.in_features
        m.head = nn.Linear(in_features, N_CLASSES)
    elif model_name == 'efficientnet_v2_s':
        try:
            m = models.efficientnet_v2_s(weights=models.EfficientNet_V2_S_Weights.IMAGENET1K_V1)
        except Exception as e:
            print('Warning: efficientnet_v2_s pretrained unavailable, fallback to random init:', e)
            m = models.efficientnet_v2_s(weights=None)
        in_features = m.classifier[1].in_features
        m.classifier[1] = nn.Linear(in_features, N_CLASSES)
    elif model_name == 'convnext_small':
        try:
            m = models.convnext_small(weights=models.ConvNeXt_Small_Weights.IMAGENET1K_V1)
        except Exception as e:
            print('Warning: convnext_small pretrained unavailable, fallback to random init:', e)
            m = models.convnext_small(weights=None)
        in_features = m.classifier[2].in_features
        m.classifier[2] = nn.Linear(in_features, N_CLASSES)
    else:
        raise ValueError(model_name)
    return m.to(device)


def set_head_only(model, model_name, head_only):
    for p in model.parameters():
        p.requires_grad = not head_only

    if head_only:
        if model_name == 'swin_v2_t':
            for p in model.head.parameters():
                p.requires_grad = True
        elif model_name == 'efficientnet_v2_s':
            for p in model.classifier[1].parameters():
                p.requires_grad = True
        elif model_name == 'convnext_small':
            for p in model.classifier[2].parameters():
                p.requires_grad = True


def pad_or_crop_to_size(x, target_h, target_w):
    _, _, h, w = x.shape
    if h > target_h:
        top = (h - target_h) // 2
        x = x[:, :, top:top + target_h, :]
    elif h < target_h:
        p = target_h - h
        x = F.pad(x, (0, 0, p // 2, p - p // 2), mode='reflect')

    _, _, h, w = x.shape
    if w > target_w:
        left = (w - target_w) // 2
        x = x[:, :, :, left:left + target_w]
    elif w < target_w:
        p = target_w - w
        x = F.pad(x, (p // 2, p - p // 2, 0, 0), mode='reflect')

    return x


def scale_view(x, scale):
    b, c, h, w = x.shape
    y = F.interpolate(x, scale_factor=scale, mode='bilinear', align_corners=False)
    y = pad_or_crop_to_size(y, h, w)
    return y


@torch.no_grad()
def tta_logits_8(model, x):
    views = [
        x,
        torch.flip(x, dims=[3]),
        torch.flip(x, dims=[2]),
        torch.flip(x, dims=[2, 3]),
        scale_view(x, 0.92),
        scale_view(x, 1.08),
        torch.flip(scale_view(x, 0.92), dims=[3]),
        torch.flip(scale_view(x, 1.08), dims=[2]),
    ]
    logits = 0
    for v in views:
        logits = logits + model(v)
    return logits / len(views)


@torch.no_grad()
def predict_proba(model, loader, use_tta=True):
    model.eval()
    out = []
    for batch in loader:
        images = batch[0].to(device, non_blocking=True)
        logits = tta_logits_8(model, images) if use_tta else model(images)
        probs = torch.softmax(logits, dim=1).cpu().numpy()
        out.append(probs)
    return np.concatenate(out, axis=0)


def macro_f1_from_proba(y_true, proba):
    pred = np.argmax(proba, axis=1)
    return f1_score(y_true, pred, average='macro')


def optimize_member_weights(y_true, member_oof, base_weights=None, n_trials=2500, seed=42):
    rng = np.random.default_rng(seed)
    n = len(member_oof)

    if base_weights is None:
        base_weights = np.ones(n, dtype=np.float32) / n
    else:
        base_weights = np.array(base_weights, dtype=np.float32)
        base_weights = base_weights / base_weights.sum()

    best_w = base_weights.copy()
    blend = np.zeros_like(member_oof[0])
    for w, p in zip(best_w, member_oof):
        blend += w * p
    best_score = macro_f1_from_proba(y_true, blend)

    for _ in range(n_trials):
        w = rng.dirichlet(np.ones(n)).astype(np.float32)
        cand = np.zeros_like(member_oof[0])
        for wi, pi in zip(w, member_oof):
            cand += wi * pi
        s = macro_f1_from_proba(y_true, cand)
        if s > best_score:
            best_score = s
            best_w = w

    return best_w, float(best_score)


def optimize_bias_temp(y_true, proba, rounds=4):
    eps = 1e-8
    logp = np.log(np.clip(proba, eps, 1.0))

    bias = np.zeros(N_CLASSES, dtype=np.float32)
    temp = 1.0

    def apply(logp, bias, temp):
        z = (logp + bias[None, :]) / temp
        z = z - z.max(axis=1, keepdims=True)
        e = np.exp(z)
        return e / e.sum(axis=1, keepdims=True)

    cur = apply(logp, bias, temp)
    best = macro_f1_from_proba(y_true, cur)

    bias_steps = [0.25, 0.12, 0.06, 0.03]
    temp_steps = [0.15, 0.08, 0.04]

    for _ in range(rounds):
        improved = False

        for c in range(N_CLASSES):
            for step in bias_steps:
                for d in (-step, step):
                    b2 = bias.copy()
                    b2[c] += d
                    p2 = apply(logp, b2, temp)
                    s = macro_f1_from_proba(y_true, p2)
                    if s > best:
                        best = s
                        bias = b2
                        improved = True

        for step in temp_steps:
            for d in (-step, step):
                t2 = max(0.55, min(1.75, temp + d))
                p2 = apply(logp, bias, t2)
                s = macro_f1_from_proba(y_true, p2)
                if s > best:
                    best = s
                    temp = t2
                    improved = True

        if not improved:
            break

    calibrated = apply(logp, bias, temp)
    return bias, float(temp), calibrated, float(best)


def apply_bias_temp(proba, bias, temp):
    eps = 1e-8
    z = (np.log(np.clip(proba, eps, 1.0)) + bias[None, :]) / temp
    z = z - z.max(axis=1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)



In [ ]:
# =========================
# Train all members (seed x model x fold)
# =========================
y_true = train_df['label'].values
member_oof_list = []
member_test_list = []
member_names = []
logs = []

for seed in SEEDS:
    seed_everything(seed)
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=seed)

    for mcfg in MODEL_CONFIGS:
        model_name = mcfg['name']
        print('')
        print('Member start:', model_name, 'seed=', seed)

        oof_member = np.zeros((len(train_df), N_CLASSES), dtype=np.float32)
        test_member = np.zeros((len(test_df), N_CLASSES), dtype=np.float32)

        for fold, (tr_idx, va_idx) in enumerate(skf.split(train_df, train_df['label']), start=1):
            print('')
            print('Fold', fold, '/', N_SPLITS)

            tr_df = train_df.iloc[tr_idx].copy()
            va_df = train_df.iloc[va_idx].copy()
            y_val = va_df['label'].values

            tr_ds = SkinDataset(tr_df, TRAIN_IMG_DIR, transform=train_tfms, is_test=False)
            va_ds = SkinDataset(va_df, TRAIN_IMG_DIR, transform=valid_tfms, is_test=False)
            te_ds = SkinDataset(test_df, TEST_IMG_DIR, transform=valid_tfms, is_test=True)

            tr_loader = build_loader(tr_ds, shuffle=True)
            va_loader = build_loader(va_ds, shuffle=False)
            te_loader = build_loader(te_ds, shuffle=False)

            model = build_model(model_name)
            class_weights = make_class_weights(tr_df).to(device)
            criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=LABEL_SMOOTHING)

            set_head_only(model, model_name, head_only=True)
            opt_head = AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=LR_HEAD, weight_decay=WEIGHT_DECAY)

            set_head_only(model, model_name, head_only=False)
            opt_full = AdamW(model.parameters(), lr=LR_FINE, weight_decay=WEIGHT_DECAY)
            sched = CosineAnnealingLR(opt_full, T_max=max(EPOCHS - HEAD_EPOCHS, 1), eta_min=1e-6)

            scaler = torch.amp.GradScaler('cuda', enabled=USE_AMP)

            best_state = None
            best_f1 = -1.0
            bad = 0

            for epoch in range(1, EPOCHS + 1):
                model.train()
                losses = []

                if epoch <= HEAD_EPOCHS:
                    set_head_only(model, model_name, head_only=True)
                    optimizer = opt_head
                else:
                    set_head_only(model, model_name, head_only=False)
                    optimizer = opt_full

                for images, labels in tr_loader:
                    images = images.to(device, non_blocking=True)
                    labels = labels.to(device, non_blocking=True)

                    optimizer.zero_grad(set_to_none=True)
                    with torch.amp.autocast('cuda', enabled=USE_AMP):
                        logits = model(images)
                        loss = criterion(logits, labels)

                    scaler.scale(loss).backward()
                    if USE_AMP:
                        scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
                    scaler.step(optimizer)
                    scaler.update()
                    losses.append(loss.item())

                if epoch > HEAD_EPOCHS:
                    sched.step()

                val_proba = predict_proba(model, va_loader, use_tta=False)
                val_f1 = macro_f1_from_proba(y_val, val_proba)
                val_pred = np.argmax(val_proba, axis=1)
                dist = pd.Series(val_pred).value_counts(normalize=True).sort_index()
                dist_dict = {idx2label[i]: round(float(dist.get(i, 0.0)), 3) for i in range(N_CLASSES)}

                print('Epoch', epoch, '/', EPOCHS, 'train_loss=', round(float(np.mean(losses)), 4), 'val_f1=', round(float(val_f1), 4), 'dist=', dist_dict)
                if max(dist_dict.values()) > 0.92:
                    print('Warning: possible collapse tendency.')

                if val_f1 > best_f1:
                    best_f1 = val_f1
                    best_state = copy.deepcopy(model.state_dict())
                    bad = 0
                else:
                    bad += 1

                if bad >= EARLY_STOP:
                    print('Early stop at epoch', epoch)
                    break

            model.load_state_dict(best_state)

            va_prob_tta = predict_proba(model, va_loader, use_tta=True)
            te_prob_tta = predict_proba(model, te_loader, use_tta=True)

            oof_member[va_idx] = va_prob_tta
            test_member += te_prob_tta / N_SPLITS

            ckpt = run_dir / f'member_{model_name}_seed{seed}_fold{fold}.pt'
            torch.save(best_state, ckpt)

            logs.append({
                'seed': seed,
                'model': model_name,
                'fold': fold,
                'best_val_f1': float(best_f1),
                'ckpt': str(ckpt),
            })

            del model, tr_ds, va_ds, te_ds, tr_loader, va_loader, te_loader
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        member_oof_list.append(oof_member)
        member_test_list.append(test_member)
        member_names.append(f'{model_name}_seed{seed}')

print('')
print('Total members:', len(member_names))


In [ ]:
# =========================
# OOF weight optimization + calibration
# =========================
base_w = []
for s in SEEDS:
    for m in MODEL_CONFIGS:
        base_w.append(m['base_weight'])
base_w = np.array(base_w, dtype=np.float32)
base_w = base_w / base_w.sum()

best_w, best_oof_before_cal = optimize_member_weights(
    y_true=y_true,
    member_oof=member_oof_list,
    base_weights=base_w,
    n_trials=3000,
    seed=123,
)

print('Best OOF macro F1 before calibration:', round(best_oof_before_cal, 6))
for n, w in zip(member_names, best_w):
    print(n, 'weight=', round(float(w), 4))

blend_oof = np.zeros_like(member_oof_list[0])
blend_test = np.zeros_like(member_test_list[0])
for w, poof, ptest in zip(best_w, member_oof_list, member_test_list):
    blend_oof += w * poof
    blend_test += w * ptest

bias_vec, temp_scalar, blend_oof_cal, best_oof_after_cal = optimize_bias_temp(y_true, blend_oof, rounds=5)
blend_test_cal = apply_bias_temp(blend_test, bias_vec, temp_scalar)

print('Calibrated OOF macro F1:', round(best_oof_after_cal, 6))
print('Bias:', bias_vec)
print('Temperature:', temp_scalar)

pred_oof = np.argmax(blend_oof_cal, axis=1)
print('')
print(classification_report(
    y_true,
    pred_oof,
    labels=list(range(N_CLASSES)),
    target_names=CLASS_NAMES,
    digits=4,
    zero_division=0,
))

cm = confusion_matrix(y_true, pred_oof, labels=list(range(N_CLASSES)))
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.title('OOF Confusion Matrix (Calibrated Blend)')
plt.xlabel('Pred')
plt.ylabel('True')
plt.tight_layout()
plt.savefig(run_dir / 'oof_confusion_matrix_calibrated.png', dpi=180)
plt.show()


In [ ]:
# =========================
# Optional conservative pseudo-label adjustment
# =========================
final_test_proba = blend_test_cal.copy()

if ENABLE_PSEUDO:
    conf = final_test_proba.max(axis=1)
    pseudo_idx = np.where(conf >= PSEUDO_THRESHOLD)[0]
    print('Pseudo selected:', len(pseudo_idx), '/', len(test_df))

    if len(pseudo_idx) >= PSEUDO_MIN_COUNT:
        pseudo_labels = np.argmax(final_test_proba[pseudo_idx], axis=1)

        # Conservative prior correction from pseudo set
        pseudo_prior = np.bincount(pseudo_labels, minlength=N_CLASSES).astype(np.float64)
        pseudo_prior = pseudo_prior / pseudo_prior.sum()

        train_prior = train_df['label'].value_counts(normalize=True).sort_index().values.astype(np.float64)
        ratio = np.clip(train_prior / np.clip(pseudo_prior, 1e-6, 1.0), 0.5, 2.0)
        ratio = ratio / ratio.mean()

        adjusted = final_test_proba * ratio[None, :]
        adjusted = adjusted / adjusted.sum(axis=1, keepdims=True)

        final_test_proba = (1.0 - PSEUDO_ALPHA) * final_test_proba + PSEUDO_ALPHA * adjusted

        pseudo_df = test_df.iloc[pseudo_idx].copy()
        pseudo_df['pseudo_label'] = pseudo_labels
        pseudo_df['pseudo_disease'] = [idx2label[int(x)] for x in pseudo_labels]
        pseudo_df['confidence'] = conf[pseudo_idx]
        pseudo_df.to_csv(run_dir / 'pseudo_selected.csv', index=False)
    else:
        print('Pseudo count below minimum; skip pseudo adjustment.')


In [ ]:
# =========================
# Final submission and artifacts
# =========================
final_pred = np.argmax(final_test_proba, axis=1)
submission = pd.DataFrame({
    'id': test_df['id'].values,
    'disease': [idx2label[int(i)] for i in final_pred],
}).sort_values('id').reset_index(drop=True)

assert len(submission) == 180
assert submission['disease'].isin(CLASS_NAMES).all()

run_submission_path = run_dir / 'submission.csv'
platform_submission_path = OUTPUT_DIR / 'submission.csv'
submission.to_csv(run_submission_path, index=False)
submission.to_csv(platform_submission_path, index=False)

if MODE == 'local':
    local_submission_path = BASE_DIR / 'submission.csv'
    submission.to_csv(local_submission_path, index=False)

pd.DataFrame(logs).to_csv(run_dir / 'member_logs.csv', index=False)

summary = {
    'run_name': run_name,
    'mode': MODE,
    'device': str(device),
    'seeds': SEEDS,
    'models': MODEL_CONFIGS,
    'member_names': member_names,
    'optimized_member_weights': [float(x) for x in best_w],
    'oof_macro_before_calibration': float(best_oof_before_cal),
    'oof_macro_after_calibration': float(best_oof_after_cal),
    'bias_vector': [float(x) for x in bias_vec],
    'temperature': float(temp_scalar),
    'pseudo_enabled': bool(ENABLE_PSEUDO),
    'pseudo_threshold': float(PSEUDO_THRESHOLD),
    'pseudo_min_count': int(PSEUDO_MIN_COUNT),
    'pseudo_alpha': float(PSEUDO_ALPHA),
}
with open(run_dir / 'run_summary.json', 'w', encoding='utf-8') as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print('Run dir:', run_dir)
print('Run submission:', run_submission_path)
print('Platform output:', platform_submission_path)
if MODE == 'local':
    print('Local copy:', local_submission_path)

print('')
print('Submission distribution:')
print(submission['disease'].value_counts())
submission.head()
